In [2]:
import sys
sys.executable

'C:\\Users\\91930\\anaconda3\\envs\\practice_env\\python.exe'

In [3]:
import os
from pathlib import Path

from PyPDF2 import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from transformers import pipeline

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [4]:
pdf_path = "../data/pdfs/sample.pdf"

def load_pdf(file_path):
    """Extract text from PDF"""
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

print(f"PDF loader function ready")
print(f"Place PDFs in: {Path('../data/pdfs').absolute()}")

PDF loader function ready
Place PDFs in: C:\Users\91930\Desktop\Python AI projects\Major Projects\research-papers-qa-rag\notebooks\..\data\pdfs


In [5]:
pdf_text = load_pdf("../data/pdfs/bert_paper.pdf")

print(f"PDF loaded successfully!")
print(f"Total characters: {len(pdf_text)}")
print(f"\nFirst 500 characters:\n{pdf_text[:500]}")

PDF loaded successfully!
Total characters: 64048

First 500 characters:
BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova
Google AI Language
fjacobdevlin,mingweichang,kentonl,kristout g@google.com
Abstract
We introduce a new language representa-
tion model called BERT , which stands for
Bidirectional Encoder Representations from
Transformers. Unlike recent language repre-
sentation models (Peters et al., 2018a; Rad-
ford et al., 2018), BERT is designed to pre-
train deep bidirec


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        
    chunk_overlap=50,      
    length_function=len,
)

chunks = text_splitter.split_text(pdf_text)

print(f"Total chunks created: {len(chunks)}")
print(f"\nFirst chunk:\n{chunks[0]}")
print(f"\nSecond chunk:\n{chunks[1]}")

Total chunks created: 145

First chunk:
BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova
Google AI Language
fjacobdevlin,mingweichang,kentonl,kristout g@google.com
Abstract
We introduce a new language representa-
tion model called BERT , which stands for
Bidirectional Encoder Representations from
Transformers. Unlike recent language repre-
sentation models (Peters et al., 2018a; Rad-
ford et al., 2018), BERT is designed to pre-

Second chunk:
ford et al., 2018), BERT is designed to pre-
train deep bidirectional representations from
unlabeled text by jointly conditioning on both
left and right context in all layers. As a re-
sult, the pre-trained BERT model can be ﬁne-
tuned with just one additional output layer
to create state-of-the-art models for a wide
range of tasks, such as question answering and
language inference, without substantial task-
speciﬁc architecture modiﬁcations.
BERT is conceptu

In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

test_embedding = embeddings.embed_query(chunks[0])

print(f"✅ Embedding model loaded")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 10 values: {test_embedding[:10]}")

C:\Users\91930\AppData\Local\Temp\ipykernel_14244\609315228.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded
Embedding dimension: 384
First 10 values: [-0.15979138016700745, -0.09003356099128723, 0.06607011705636978, -0.002907665679231286, -0.07875494658946991, 0.07705871015787125, -0.021570388227701187, 0.03649529814720154, 0.03297724947333336, -0.04337286949157715]


In [8]:
vectorstore = FAISS.from_texts(
    texts=chunks,
    embedding=embeddings
)

print(f"✅ FAISS vector store created")
print(f"Total vectors stored: {vectorstore.index.ntotal}")

✅ FAISS vector store created
Total vectors stored: 145


In [10]:
question = "What is BERT?"

retrieved_docs = vectorstore.similarity_search(question, k=3)

print(f"Question: {question}\n")
print(f"Retrieved {len(retrieved_docs)} chunks:\n")

for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content[:200]) 
    print()

Question: What is BERT?

Retrieved 3 chunks:

--- Chunk 1 ---
BERT is conceptually simple and empirically
powerful. It obtains new state-of-the-art re-
sults on eleven natural language processing
tasks, including pushing the GLUE score to
80.5% (7.7% point absol

--- Chunk 2 ---
question-answering example in Figure 1 will serve
as a running example for this section.
A distinctive feature of BERT is its uniﬁed ar-
chitecture across different tasks. There is mini-mal difference

--- Chunk 3 ---
between how BERT and GPT were trained:
• GPT is trained on the BooksCorpus (800M
words); BERT is trained on the BooksCor-
pus (800M words) and Wikipedia (2,500M
words).
• GPT uses a sentence separator



first 200 characters

In [12]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=512,
    device=-1  
)

print("✅ LLM loaded")

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCa

✅ LLM loaded


In [13]:
def answer_question(question):
    """Complete RAG pipeline"""
    
    retrieved_docs = vectorstore.similarity_search(question, k=3)
    
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    prompt = f"""Answer the question based on the context below.
    
Context: {context}

Question: {question}

Answer:"""
    
    result = llm(prompt, max_length=200, do_sample=False)
    answer = result[0]['generated_text']
    
    return answer, retrieved_docs

question = "What is BERT used for?"
answer, sources = answer_question(question)

print(f"Question: {question}\n")
print(f"Answer: {answer}\n")
print(f"\nSources (first 100 chars of each):")
for i, doc in enumerate(sources):
    print(f"{i+1}. {doc.page_content[:100]}...")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Question: What is BERT used for?

Answer: Answer the question based on the context below.

Context: BERT is conceptually simple and empirically
powerful. It obtains new state-of-the-art re-
sults on eleven natural language processing
tasks, including pushing the GLUE score to
80.5% (7.7% point absolute improvement),
MultiNLI accuracy to 86.7% (4.6% absolute
improvement), SQuAD v1.1 question answer-
ing Test F1 to 93.2 (1.5 point absolute im-
provement) and SQuAD v2.0 Test F1 to 83.1
(5.1 point absolute improvement).
1 Introduction
Language model pre-training has been shown to

question-answering example in Figure 1 will serve
as a running example for this section.
A distinctive feature of BERT is its uniﬁed ar-
chitecture across different tasks. There is mini-mal difference between the pre-trained architec-
ture and the ﬁnal downstream architecture.
Model Architecture BERT’s model architec-
ture is a multi-layer bidirectional Transformer en-
coder based on the original implementation d